<a href="https://colab.research.google.com/github/hsavthegreat/eyeclip/blob/main/notebooks/eyeclip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EyeCLIP: Few-Shot Retinal Fundus Classification Notebook
This notebook demonstrates a complete end-to-end framework for few-shot learning and zero-shot evaluations on the RFMiD2.0 dataset using ResNet-50, CLIP, and BiomedCLIP encoders.

## Step 1: Environment Setup & Directory Mount
Mounting Google Drive as our persistent workspace to store raw data, extracted embeddings, and results. This corresponds to the drive mounting block (Cell 1).

In [32]:
# ═════════════════════════════════════════════════════════════════
# Cell 1 — Mount Drive and create eyeclip/{data,embeddings,results}
# Drive is the only persistent store; none of it is ever committed.
# ═════════════════════════════════════════════════════════════════
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")              # approve the OAuth prompt

ROOT = Path("/content/drive/MyDrive/eyeclip")
for sub in ("data", "embeddings", "results"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

# ── sanity check: all three dirs exist, are writable, entry counts ──
for sub in ("data", "embeddings", "results"):
    p = ROOT / sub
    probe = p / ".write_test"
    probe.write_text("ok"); probe.unlink()          # write + delete probe
    print(f"{p}  exists={p.is_dir()}  writable=True  entries={len(list(p.iterdir()))}")
print("\nReady. Datasets go under:", ROOT / "data")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/eyeclip/data  exists=True  writable=True  entries=1
/content/drive/MyDrive/eyeclip/embeddings  exists=True  writable=True  entries=8
/content/drive/MyDrive/eyeclip/results  exists=True  writable=True  entries=3

Ready. Datasets go under: /content/drive/MyDrive/eyeclip/data


## Step 2: Dependencies Installation
Installing and verifying essential library dependencies like `open_clip_torch`, `transformers`, and `scikit-learn` (Cell 2).

In [39]:
# ═════════════════════════════════════════════════════════════════
# Cell 2 — Install the Stage-1..3 dependency list.
# Colab preinstalls most of these: pip staying quiet / "already
# satisfied" is EXPECTED, not an error.
# ═════════════════════════════════════════════════════════════════
%pip install -q open_clip_torch transformers scikit-learn pandas matplotlib pillow opencv-python

# ── sanity check: every package imports; versions printed ──
import importlib.metadata as md
import numpy, pandas, sklearn, matplotlib, PIL, transformers, open_clip

print("── package versions (VERIFY: whatever your Colab runtime provides) ──")
for p in ["open_clip_torch", "transformers", "scikit-learn", "pandas",
          "matplotlib", "pillow", "opencv-python", "numpy"]:
    try:
        print(f"{p:<16} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:<16} MISSING")

# opencv-python (GUI build) can clash with Colab's preinstalled headless
# build and fail with a libGL error. If so, switch to headless (same cv2
# API for our purposes — we never call cv2.imshow):
try:
    import cv2
    print(f"\ncv2 imports fine: {cv2.__version__}")
except Exception as e:
    print(f"\ncv2 import failed ({e}) -> reinstalling headless build ...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "opencv-python", "opencv-python-headless",
                    "opencv-contrib-python"], capture_output=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless"], capture_output=True)
    import cv2
    print(f"cv2 after headless fix: {cv2.__version__}")

# ── GPU sanity (needed only in Stage 2 for the frozen encoders) ──
import torch
print(f"\ntorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

── package versions (VERIFY: whatever your Colab runtime provides) ──
open_clip_torch  4.2.1
transformers     5.22.1
scikit-learn     1.6.1
pandas           2.2.3
matplotlib       3.10.0
pillow           11.3.0
opencv-python    5.0.0.93
numpy            2.4.6

cv2 imports fine: 4.14.0

torch 2.14.1+cu130 | CUDA available: True
device: Tesla T4


## Step 3: Dataset Caching & Manifest Generation
Parsing label CSV files, matching them with local fundus images across splits, and setting up support dataset samplings (Cell 3).

In [34]:
# ============================================================================
# Cell 3 (v3.1) — Cache manifest + experiment manifest (APPROVED protocol)
# ============================================================================
import json
import hashlib
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

# --- CONFIG -----------------------------------------------------------------
DATA_ROOT = Path("/content/drive/MyDrive/eyeclip/data/dataset_main/RFMiD2_0")  # NEW path
OUT_DIR   = Path("/content/drive/MyDrive/eyeclip/manifests")

SEEDS = list(range(10))            # approved (2)
SHOTS = [1, 5, 10]
MIN_TOTAL_1POS = 20                # approved: >= 20 exactly-one-positive per class (all splits)
MIN_TRAIN_1POS = max(SHOTS)        # support is train-only, so 10-shot needs >= 10 train images
SECONDARY_CLASSES     = ["BRVO"]   # approved (4): secondary table only
FORCE_INCLUDE_CLASSES = []         # e.g. ["MH"] — force a class into MAIN despite the rules
EXPECTED_CACHE_N = 856             # approved (5): soft check (509 + 177 + 170)

IMG_COL_OVERRIDE = "ID"
MANUAL_SPLIT_MAP = {}
# -----------------------------------------------------------------------------
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# [1] Discover CSVs and images (global; duplicate stems resolved) ---------------
csv_paths = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() == ".csv")
img_paths = sorted(p for p in DATA_ROOT.rglob("*")
                   if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"})
assert csv_paths and img_paths, "No CSVs or images found — check DATA_ROOT."

EXT_PREF = {".png": 0, ".jpg": 1, ".jpeg": 2}
candidates = {}
for p in img_paths:
    candidates.setdefault(p.stem.lower(), []).append(p)

stem_to_path, dup_groups = {}, []
for k, ps in candidates.items():
    if len(ps) == 1:
        stem_to_path[k] = ps[0]
        continue
    ps_sorted = sorted(ps, key=lambda q: (EXT_PREF.get(q.suffix.lower(), 9), q.name))
    chosen = ps_sorted[0]
    stem_to_path[k] = chosen
    h0 = hashlib.md5(chosen.read_bytes()).hexdigest()
    same = all(hashlib.md5(q.read_bytes()).hexdigest() == h0 for q in ps_sorted[1:])
    dup_groups.append((k, [q.name for q in ps_sorted], chosen.name, same))

print(f"[1] {len(img_paths)} image files, {len(stem_to_path)} unique stems")
if dup_groups:
    print(f"    NOTE: {len(dup_groups)} duplicate-stem group(s), resolved (prefer .png):")
    for k, names, chosen, same in dup_groups:
        print(f"      {k}: {names} -> using {chosen} | byte-identical copies: {same}")

def guess_split(s):
    s = s.lower()
    if "train" in s: return "train"
    if "test" in s:  return "test"
    if "valid" in s: return "validation"
    if "val" in s and "eval" not in s: return "validation"
    return None

split_csv = {}
for p in csv_paths:
    mapped = next((v for k, v in MANUAL_SPLIT_MAP.items() if k.lower() == p.name.lower()), None)
    if mapped is None:
        mapped = guess_split(p.name) or guess_split(str(p.relative_to(DATA_ROOT)))
    assert mapped in ("train", "validation", "test"), \
        f"Cannot map {p.relative_to(DATA_ROOT)} to a split — set MANUAL_SPLIT_MAP."
    assert mapped not in split_csv, f"Two CSVs map to '{mapped}': {split_csv[mapped].name}, {p.name}"
    split_csv[mapped] = p
missing = {"train", "validation", "test"} - set(split_csv)
assert not missing, f"Missing split CSVs for: {missing}"
print("    split CSVs: " + ", ".join(f"{k} <- {v.name}" for k, v in sorted(split_csv.items())))

# [2] Helpers --------------------------------------------------------------------
def find_binary_cols(df):
    out = []
    for c in df.columns:
        v = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(v) and set(v.unique()) <= {0.0, 1.0}:
            out.append(c)
    return out

def stem_of(v):
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    s = str(v).strip().lower()
    for ext in (".png", ".jpg", ".jpeg"):
        if s.endswith(ext):
            s = s[: -len(ext)]
    return s

def safe_read_csv(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Unable to decode {path} using standard encodings.")

# [3] Match label rows to images — FOLDER-AWARE (a row only matches a file in ITS OWN split folder)
img_split = {k: guess_split(str(p.relative_to(DATA_ROOT).parent)) for k, p in stem_to_path.items()}
FOLDER_AWARE = all(v is not None for v in img_split.values())
print("\n[3a] image files per folder-split:",
      pd.Series(img_split).fillna("UNKNOWN").value_counts().to_dict(),
      "| folder-aware matching:", FOLDER_AWARE)

rows, stats, unmatched_log, label_cols_ref, id_sets = [], {}, [], None, {}
for split in ("train", "validation", "test"):
    df = safe_read_csv(split_csv[split])
    df.columns = [str(c).strip() for c in df.columns]
    binary = find_binary_cols(df)
    label_cols = [c for c in binary if "risk" not in c.lower()]
    if label_cols_ref is None:
        label_cols_ref = label_cols
    assert set(label_cols) == set(label_cols_ref), f"[{split}] label columns differ from train"
    img_col = IMG_COL_OVERRIDE
    assert img_col in df.columns, f"[{split}] IMG_COL_OVERRIDE '{img_col}' not in columns"
    ids = df[img_col].map(stem_of)
    assert ids.is_unique, f"[{split}] duplicate image IDs in {split_csv[split].name}"
    id_sets[split] = set(ids)
    pos   = df[label_cols].to_numpy(dtype=int)
    n_pos = pos.sum(axis=1)
    n_match = n_unmatch = n_1pos = 0
    for i in range(len(df)):
        stem = ids.iloc[i]
        ok = stem in stem_to_path and (not FOLDER_AWARE or img_split[stem] == split)
        if not ok:
            n_unmatch += 1
            unmatched_log.append((split, stem, img_split.get(stem, "no file on disk")))
            continue
        n_match += 1
        if n_pos[i] == 1:
            n_1pos += 1
            cls = label_cols[int(pos[i].argmax())]
        else:
            cls = ""
        rows.append((stem, str(stem_to_path[stem].relative_to(DATA_ROOT)), split,
                     int(n_pos[i]), cls))
    stats[split] = (len(df), n_match, n_unmatch, n_1pos)

print(f"\n[3] Row-to-image matching per split (rows, matched, unmatched, 1pos&matched) "
      f"— image column: '{IMG_COL_OVERRIDE}'")
for k, (nr, nm, nu, n1) in stats.items():
    print(f"      {k:<11} {nr:>5} {nm:>6} {nu:>6} {n1:>6}")
print("\n[3b] Unmatched label rows (split, ID, where a file with that ID actually lives):")
for u in unmatched_log:
    print("      ", u)
print("[3c] IDs listed in more than one split CSV:",
      sorted((id_sets["train"] & id_sets["validation"]) |
             (id_sets["train"] & id_sets["test"]) |
             (id_sets["validation"] & id_sets["test"])))

m = pd.DataFrame(rows, columns=["stem", "rel_path", "orig_split", "n_pos", "single_class"])
dup = m[m["stem"].duplicated(keep=False)]
if len(dup):
    print("\n!! still duplicated after folder-aware matching:")
    print(dup.sort_values("stem").to_string())
assert m["stem"].is_unique, "Same stem matched twice — paste the [3a]-[3c] output + table above."
m = m.sort_values(["orig_split", "stem"]).reset_index(drop=True)
one = m[m["n_pos"] == 1].copy()      # experiment layer = exactly-one-positive rows
assert len(one) > 0, "No matched exactly-one-positive images."

# [4] Class table + roles (from matched 1pos rows — approved logic, unchanged) ---
ct = pd.crosstab(one["single_class"], one["orig_split"])
for col in ("train", "validation", "test"):
    if col not in ct.columns:
        ct[col] = 0
ct = ct[["train", "validation", "test"]]
ct["eval_n"] = ct["validation"] + ct["test"]
ct["total"]  = ct[["train", "validation", "test"]].sum(axis=1)
ct = ct.sort_values("total", ascending=False)

def resolve(name):
    for c in ct.index:
        if c.lower() == str(name).lower():
            return c
    raise AssertionError(f"Class {name!r} not found among the label columns.")

SECONDARY = [resolve(c) for c in SECONDARY_CLASSES]
FORCED    = [resolve(c) for c in FORCE_INCLUDE_CLASSES]

def class_role(c):
    r = ct.loc[c]
    if c in FORCED:    return "main-forced"
    if c in SECONDARY: return "secondary"
    if r["total"] >= MIN_TOTAL_1POS and r["train"] >= MIN_TRAIN_1POS and r["eval_n"] >= 1:
        return "main"
    if r["total"] >= MIN_TOTAL_1POS:
        return "flagged"
    return "unused"

ct["role"] = [class_role(c) for c in ct.index]
main = sorted(c for c in ct.index if ct.loc[c, "role"].startswith("main"))
assert main, "No MAIN classes — check MIN_TOTAL_1POS / SECONDARY_CLASSES / FORCE_INCLUDE_CLASSES."

print(f"\n[4] Class table ({len(ct)} classes): "
      f"{(ct['role'] == 'main').sum()} main, {(ct['role'] == 'secondary').sum()} secondary, "
      f"{(ct['role'] == 'flagged').sum()} flagged, {(ct['role'] == 'unused').sum()} unused")
hdr = f"{'class':<24}{'train':>6}{'val':>5}{'test':>5}{'eval':>6}{'total':>6}  role"
print(hdr); print("-" * len(hdr))
for c, r in ct.iterrows():
    note = ""
    if r["role"] == "flagged":
        why = []
        if r["train"] < MIN_TRAIN_1POS: why.append(f"train {int(r['train'])} < {MIN_TRAIN_1POS}")
        if r["eval_n"] < 1:             why.append("no eval images")
        note = "   <- " + "; ".join(why)
    print(f"{c:<24}{int(r['train']):>6}{int(r['validation']):>5}{int(r['test']):>5}"
          f"{int(r['eval_n']):>6}{int(r['total']):>6}  {r['role']}{note}")

print(f"\n[5] MAIN classes ({len(main)}) — support pool = train, evaluation = validation + test:")
for c in main:
    r = ct.loc[c]
    flag = "  <-- small eval set: recall will be fragile" if r["eval_n"] < 10 else ""
    print(f"      {c:<10} train {int(r['train']):>4}   eval {int(r['eval_n']):>4}{flag}")

# [6] Support assignments (approved logic: same seeds, same pools as v1/v2) -------
train_stems = {c: sorted(one.loc[(one["single_class"] == c) & (one["orig_split"] == "train"), "stem"])
               for c in ct.index}
eval_stems  = {c: sorted(one.loc[(one["single_class"] == c) & (one["orig_split"] != "train"), "stem"])
               for c in ct.index}

sup_rows = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    for c in main:
        order = list(rng.permutation(train_stems[c]))
        for shot in SHOTS:
            sup_rows.append({
                "seed": seed,
                "class": c,
                "shot": shot,
                "support": ";".join(sorted(order[:shot]))
            })
sup = pd.DataFrame(sup_rows)

n_bad = 0
for _, r in sup.iterrows():
    s = set(r["support"].split(";"))
    if len(s) != r["shot"]:                     n_bad += 1
    if not s <= set(train_stems[r["class"]]):   n_bad += 1
    if s & set(eval_stems[r["class"]]):         n_bad += 1
assert n_bad == 0, f"{n_bad} support-assignment violations"
nest_ok = all(
    set(sup[(sup.seed == sd) & (sup["class" ] == c) & (sup.shot == 1)].iloc[0]["support"].split(";"))
    <= set(sup[(sup.seed == sd) & (sup["class"] == c) & (sup.shot == 5)].iloc[0]["support"].split(";"))
    <= set(sup[(sup.seed == sd) & (sup["class"] == c) & (sup.shot == 10)].iloc[0]["support"].split(";"))
    for sd in SEEDS for c in main)
print(f"\n[6] Support assignments: {len(sup)} rows "
      f"({len(SEEDS)} seeds x {len(main)} classes x {len(SHOTS)} shots)")
print("      all supports: from train only, disjoint from eval, correct size ... True")
print(f"      nested across shots (1 inside 5 inside 10) ... {nest_ok}")

# [7] Save to Drive (never committed) ---------------------------------------------
old = OUT_DIR / "manifest_single_label.csv"       # v1 output — superseded
if old.exists():
    old.unlink()
    print("      (removed superseded v1 file: manifest_single_label.csv)")

protocol = {
    "written": date.today().isoformat(),
    "image_id_column": IMG_COL_OVERRIDE,
    "label_protocol": "experiments use exactly-one-positive rows only; single-label; "
                      "closed world over MAIN classes",
    "splits": {"support_pool": "train only", "evaluation": "validation + test combined"},
    "no_eval_tuning": "evaluation set is never used for model selection or hyperparameters",
    "hybrid_hyperparams": "hybrid blend weight and temperature fixed in advance or selected on support set only",
    "seeds": SEEDS,
    "shots": SHOTS,
    "class_eligibility": {"min_total_one_positive": MIN_TOTAL_1POS,
                          "min_train_one_positive": MIN_TRAIN_1POS},
    "main_classes": main,
    "secondary_classes": SECONDARY,
    "metrics": ["macro-F1", "balanced accuracy",
                "per-class recall (always reported together with per-class evaluation counts)"],
    "support_sampling": ("per seed: numpy random.default_rng(seed), classes in sorted order; "
                         "support = first k images of one fixed permutation (nested across shots)"),
    "cache_policy": ("embeddings cached for ALL images that have both a label row and an image "
                     "file, label-agnostic (expected 856); class filtering happens at experiment "
                     "time; stems matched globally across split folders"),
}
m.to_csv(OUT_DIR / "manifest_cache.csv", index=False)
ct.to_csv(OUT_DIR / "class_table.csv")
sup.to_csv(OUT_DIR / "support_assignments.csv", index=False)
(OUT_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2))
print("\n[7] Written to Drive:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"      {f.name:<28} {f.stat().st_size / 1e3:8.1f} kB")

# [8] Cache composition + size check vs the approved proposal ----------------------
print(f"\n[8] Cache composition (label-agnostic — ALL matched images): {len(m)} total")
for k, v in m["n_pos"].value_counts().sort_index().items():
    tag = "   <- experiment layer" if k == 1 else ""
    print(f"      n_pos = {k:>2}: {v:>5} images{tag}")
n_rows_total      = sum(s[0] for s in stats.values())
n_unmatched_total = sum(s[2] for s in stats.values())
n_orphan_files    = len(stem_to_path) - len(m)
print(f"      reconciliation: {n_rows_total} label rows - {n_unmatched_total} without image file "
      f"= {n_rows_total - n_unmatched_total};  {len(stem_to_path)} unique stems - "
      f"{n_orphan_files} with no label row = {len(m)}")
if len(m) != EXPECTED_CACHE_N:
    print(f"      !! cache = {len(m)} != expected {EXPECTED_CACHE_N} — check the per-split "
          f"unmatched counts above, paste this output back BEFORE any embeddings.")
else:
    print(f"      cache = {len(m)} == expected {EXPECTED_CACHE_N} — matches the approved proposal.")
print("\nSTOP POINT (approved): nothing has been embedded yet. Paste this full output back;")
print("once the cache count and the MAIN class list are confirmed, I'll send Cell 5.")

# Documentation helper: the 6 unmatched label rows and the orphan image files
lab = {}
for split in ("train", "validation", "test"):
    d = safe_read_csv(split_csv[split]); d.columns = [str(c).strip() for c in d.columns]
    d["_id"] = d["ID"].map(stem_of); lab[split] = d
for split, stem, where in unmatched_log:
    r = lab[split][lab[split]["_id"] == stem].iloc[0]
    print(split, stem, "-> positive labels:", [c for c in label_cols_ref if int(r[c]) == 1])
matched_stems = set(m["stem"])
print("orphan image files (no label row):",
      sorted((img_split[k], k) for k in stem_to_path if k not in matched_stems))


[1] 860 image files, 860 unique stems
    split CSVs: test <- RFMiD_2_Testing_labels.csv, train <- RFMiD_2_Training_labels.csv, validation <- RFMiD_2_Validation_labels.csv

[3a] image files per folder-split: {'train': 509, 'validation': 177, 'test': 174} | folder-aware matching: True

[3] Row-to-image matching per split (rows, matched, unmatched, 1pos&matched) — image column: 'ID'
      train         509    507      2    392
      validation    177    177      0    142
      test          174    170      4    129

[3b] Unmatched label rows (split, ID, where a file with that ID actually lives):
       ('train', '89', 'no file on disk')
       ('train', '216', 'test')
       ('test', '151', 'no file on disk')
       ('test', '191', 'validation')
       ('test', '405', 'no file on disk')
       ('test', '490', 'validation')
[3c] IDs listed in more than one split CSV: ['191', '216', '490']

[4] Class table (43 classes): 4 main, 1 secondary, 0 flagged, 38 unused
class                    tra

## Step 4: Extracting Image & Text Embeddings
Using pre-trained models (ImageNet ResNet50, OpenAI CLIP, and BiomedCLIP) to generate and save feature vectors (Cell 4).

In [35]:
# Cell 4: embeddings for all 854 images — ResNet-50 (ImageNet), CLIP ViT-B/16 (OpenAI weights), BiomedCLIP
import subprocess, sys, time, shutil, traceback
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "transformers", "timm", "ftfy"], check=True)
from pathlib import Path
import numpy as np, pandas as pd, torch, torchvision, open_clip
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path("/content/drive/MyDrive/eyeclip/data/dataset_main/RFMiD2_0")
MAN_DIR   = Path("/content/drive/MyDrive/eyeclip/manifests")
OUT       = Path("/content/drive/MyDrive/eyeclip/embeddings"); OUT.mkdir(parents=True, exist_ok=True)
LOCAL     = Path("/content/rfmid_local")
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

m = pd.read_csv(MAN_DIR / "manifest_cache.csv")
assert len(m) == 854, len(m)
if not LOCAL.exists():                       # one-time copy: Drive reads are slow
    t0 = time.time(); shutil.copytree(DATA_ROOT, LOCAL); print(f"copied images locally in {time.time()-t0:.0f}s")
paths = [LOCAL / p for p in m["rel_path"]]
assert all(p.exists() for p in paths)
m[["stem"]].to_csv(OUT / "index.csv", index=False)      # row order of every .npy

ct = pd.read_csv(MAN_DIR / "class_table.csv", index_col=0)
TEXT_CLASSES = sorted(ct.index[ct["role"].isin(["main", "secondary"])])
NAMES = {"WNL": "a normal healthy retina", "DR": "diabetic retinopathy", "CRS": "chorioretinitis",
         "RT": "retinal traction", "BRVO": "branch retinal vein occlusion"}   # edit freely
TEMPLATES = ["a fundus photograph of {}", "a retinal fundus image showing {}",
             "a color fundus photograph, {}", "a medical image of the retina: {}"]
assert set(TEXT_CLASSES) <= set(NAMES), set(TEXT_CLASSES) - set(NAMES)

class Imgs(Dataset):
    def __init__(s, paths, tf): s.paths, s.tf = paths, tf
    def __len__(s): return len(s.paths)
    def __getitem__(s, i): return s.tf(Image.open(s.paths[i]).convert("RGB"))

@torch.no_grad()
def embed_images(encode, tf, bs=32):
    out = []
    for x in DataLoader(Imgs(paths, tf), batch_size=bs, num_workers=2):
        with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            f = encode(x.to(dev))
        f = f.float(); f = f / f.norm(dim=-1, keepdim=True)
        out.append(f.cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def embed_text(encode_text):
    embs = []
    for c in TEXT_CLASSES:
        f = encode_text([t.format(NAMES[c]) for t in TEMPLATES]).float()
        f = f / f.norm(dim=-1, keepdim=True)
        f = f.mean(0); f = f / f.norm()
        embs.append(f.cpu().numpy())
    return np.stack(embs)

def load_resnet():
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1
    mdl = torchvision.models.resnet50(weights=w); mdl.fc = torch.nn.Identity()
    mdl = mdl.eval().to(dev)
    return mdl, w.transforms(), None

def load_oc(name, pretrained=None, ctx=None):
    model, _, tf = open_clip.create_model_and_transforms(name, pretrained=pretrained)
    tok = open_clip.get_tokenizer(name); model = model.eval().to(dev)
    enc_t = lambda texts: model.encode_text(tok(texts, context_length=ctx).to(dev) if ctx else tok(texts).to(dev))
    return model.encode_image, tf, enc_t

MODELS = {
    "resnet50_imagenet": lambda: (lambda mdl, tf, _: (mdl, tf, None))(*load_resnet()),
    "clip_vit_b_16_openai": lambda: load_oc("ViT-B-16-quickgelu", "openai"),
    "biomedclip_vit_b_16": lambda: load_oc("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224", None, ctx=256),
}

status = {}
for key, loader in MODELS.items():
    f_img = OUT / f"{key}.npy"
    if f_img.exists() and np.load(f_img).shape[0] == len(m):
        print(f"[skip] {key}: already saved"); status[key] = "ok (cached)"; continue
    try:
        print(f"\n=== {key} ===")
        encode, tf, enc_t = loader()
        t0 = time.time()
        E = embed_images(encode, tf)
        np.save(f_img, E)                                   # save images FIRST, so text problems can't lose them
        print(f"images: {E.shape} in {time.time()-t0:.0f}s")
        if enc_t is not None:
            T = embed_text(enc_t)
            np.savez(OUT / f"{key}_text.npz", classes=np.array(TEXT_CLASSES), emb=T)
            print("text:", T.shape, TEXT_CLASSES)
        status[key] = "ok"
    except Exception:
        traceback.print_exc(); status[key] = "FAILED (see traceback above)"
    finally:
        torch.cuda.empty_cache()

print("\n=== Verification ===")
for key in MODELS:
    f = OUT / f"{key}.npy"
    if not f.exists():
        print(f"{key}: MISSING  [{status.get(key)}]"); continue
    E = np.load(f); rs = np.random.default_rng(0).choice(len(E), 200)
    print(f"{key}: {E.shape} finite={np.isfinite(E).all()} norm~{np.linalg.norm(E,axis=1).mean():.3f} "
          f"mean pairwise cos={float((E[rs] @ E[rs].T).mean()):.3f}  [{status[key]}]")
print("STOP: paste this output back before the few-shot cell.")

device: cuda
[skip] resnet50_imagenet: already saved
[skip] clip_vit_b_16_openai: already saved
[skip] biomedclip_vit_b_16: already saved

=== Verification ===
resnet50_imagenet: (854, 2048) finite=True norm~1.000 mean pairwise cos=0.802  [ok (cached)]
clip_vit_b_16_openai: (854, 512) finite=True norm~1.000 mean pairwise cos=0.912  [ok (cached)]
biomedclip_vit_b_16: (854, 512) finite=True norm~1.000 mean pairwise cos=0.873  [ok (cached)]
STOP: paste this output back before the few-shot cell.


## Step 5: Metric Evaluation Pipeline
Running evaluations across Zero-Shot, Prototype-matching, Linear probing, and Hybrid methods over 10 randomized seeds (Cell 5).

In [36]:
# Cell 5: Few-Shot Evaluation on Cached Embeddings (CPU)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import seaborn as sns

# Constants
EMBEDDINGS_ROOT = "/content/drive/MyDrive/eyeclip/embeddings"
RESULTS_ROOT = "/content/drive/MyDrive/eyeclip/results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# Load data
manifest = pd.read_csv("/content/drive/MyDrive/eyeclip/manifests/manifest_cache.csv")
support_assignments = pd.read_csv("/content/drive/MyDrive/eyeclip/manifests/support_assignments.csv")
class_table = pd.read_csv("/content/drive/MyDrive/eyeclip/manifests/class_table.csv")

# MAIN classes only
MAIN_CLASSES = ["WNL", "DR", "CRS", "RT"]
class_to_idx = {cls: i for i, cls in enumerate(MAIN_CLASSES)}

# Get eval indices (validation + test, single-label only)
eval_indices = manifest[
    (manifest["single_class"].isin(MAIN_CLASSES)) &
    (manifest["n_pos"] == 1) &
    (manifest["orig_split"].isin(["validation", "test"]))
].index

# Get eval counts per class
eval_counts = manifest.loc[eval_indices].groupby("single_class").size().to_dict()
pd.DataFrame({"class": MAIN_CLASSES, "n_eval": [eval_counts.get(cls, 0) for cls in MAIN_CLASSES]}).to_csv(
    f"{RESULTS_ROOT}/eval_counts.csv", index=False)

# Load embeddings and text embeddings
embeddings = {
    "resnet50_imagenet": np.load(f"{EMBEDDINGS_ROOT}/resnet50_imagenet.npy"),
    "clip_vit_b_16_openai": np.load(f"{EMBEDDINGS_ROOT}/clip_vit_b_16_openai.npy"),
    "biomedclip_vit_b_16": np.load(f"{EMBEDDINGS_ROOT}/biomedclip_vit_b_16.npy")
}

text_embeddings = {
    "clip_vit_b_16_openai": np.load(f"{EMBEDDINGS_ROOT}/clip_vit_b_16_openai_text.npz"),
    "biomedclip_vit_b_16": np.load(f"{EMBEDDINGS_ROOT}/biomedclip_vit_b_16_text.npz")
}

# Precompute train split mean for centering
train_indices = manifest[manifest["orig_split"] == "train"].index
train_embeddings = {model: embeddings[model][train_indices] for model in embeddings}

# Preprocessing function
def preprocess_embeddings(embeddings, model_name, centred=True):
    if centred:
        mean_emb = np.mean(train_embeddings[model_name], axis=0)
        centered = embeddings - mean_emb
        return centered / np.linalg.norm(centered, axis=1, keepdims=True)
    return embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Get support indices for each shot/seed
def get_support_indices(shot, seed):
    seed_data = support_assignments[support_assignments["seed"] == seed]
    support_indices = []
    for _, row in seed_data.iterrows():
        class_name = row["class"]
        if class_name in MAIN_CLASSES:
            stems = row["support"].split(";")[:shot]
            support_indices.extend(manifest[manifest["stem"].astype(str).isin([str(s) for s in stems])].index)
    return support_indices

# Evaluation function
def evaluate_method(model_name, method, shot, seed, centred, embeddings, text_embeddings):
    # Get support and eval data
    support_indices = get_support_indices(shot, seed)
    eval_emb = preprocess_embeddings(embeddings[model_name][eval_indices], model_name, centred)

    # Get true labels
    true_labels = manifest.loc[eval_indices]["single_class"].map(class_to_idx)

    if method == "zero_shot":
        # Only for CLIP models
        if model_name not in text_embeddings:
            return None

        # Get text embeddings (already normalized) and map to main classes only
        text_classes_saved = list(text_embeddings[model_name]["classes"])
        main_class_indices = [text_classes_saved.index(c) for c in MAIN_CLASSES if c in text_classes_saved]
        text_emb = text_embeddings[model_name]["emb"][main_class_indices]

        # Compute cosine similarities
        sim = np.dot(eval_emb, text_emb.T)
        preds = np.argmax(sim, axis=1)

    elif method == "prototype":
        # Get support embeddings (preprocessed same way)
        support_emb = preprocess_embeddings(embeddings[model_name][support_indices], model_name, centred)

        # Compute class prototypes
        prototypes = {}
        for cls in MAIN_CLASSES:
            cls_indices = [idx for idx in support_indices if manifest.loc[idx]["single_class"] == cls]
            if len(cls_indices) > 0:
                prototypes[cls] = np.mean(support_emb[[support_indices.index(idx) for idx in cls_indices]], axis=0)

        # Compute cosine similarities to prototypes
        sim = np.zeros((len(eval_emb), len(MAIN_CLASSES)))
        for i, cls in enumerate(MAIN_CLASSES):
            if cls in prototypes:
                sim[:, i] = np.dot(eval_emb, prototypes[cls]) / (
                    np.linalg.norm(eval_emb, axis=1) * np.linalg.norm(prototypes[cls])
                )
            else:
                sim[:, i] = -np.inf

        preds = np.argmax(sim, axis=1)

    elif method == "logistic":
        # Get support labels
        support_labels = manifest.loc[support_indices]["single_class"].map(class_to_idx)

        # Train logistic regression
        clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', multi_class='multinomial')
        clf.fit(embeddings[model_name][support_indices], support_labels)

        # Predict
        preds = clf.predict(embeddings[model_name][eval_indices])

    elif method == "hybrid":
        # Only for CLIP models
        if model_name not in text_embeddings:
            return None

        # Get text embeddings filtered to MAIN_CLASSES
        text_classes_saved = list(text_embeddings[model_name]["classes"])
        main_class_indices = [text_classes_saved.index(c) for c in MAIN_CLASSES if c in text_classes_saved]
        text_emb = text_embeddings[model_name]["emb"][main_class_indices]

        # Get prototype scores
        support_emb = preprocess_embeddings(embeddings[model_name][support_indices], model_name, centred)
        prototypes = {}
        for cls in MAIN_CLASSES:
            cls_indices = [idx for idx in support_indices if manifest.loc[idx]["single_class"] == cls]
            if len(cls_indices) > 0:
                prototypes[cls] = np.mean(support_emb[[support_indices.index(idx) for idx in cls_indices]], axis=0)

        # Compute both scores
        text_sim = np.dot(eval_emb, text_emb.T)
        proto_sim = np.zeros((len(eval_emb), len(MAIN_CLASSES)))
        for i, cls in enumerate(MAIN_CLASSES):
            if cls in prototypes:
                proto_sim[:, i] = np.dot(eval_emb, prototypes[cls]) / (
                    np.linalg.norm(eval_emb, axis=1) * np.linalg.norm(prototypes[cls])
                )
            else:
                proto_sim[:, i] = -np.inf

        # Combine with alpha=0.5
        combined_scores = 0.5 * text_sim + 0.5 * proto_sim
        preds = np.argmax(combined_scores, axis=1)

    # Compute metrics
    macro_f1 = f1_score(true_labels, preds, average='macro', zero_division=0)
    balanced_acc = balanced_accuracy_score(true_labels, preds)

    return {
        "macro_f1": macro_f1,
        "balanced_acc": balanced_acc,
        "preds": preds,
        "true_labels": true_labels
    }

# Run all experiments
results = []
per_class_recalls = []

# Gather methods dynamically depending on the model
all_combinations = []
for model_name in embeddings:
    for centred in [True, False]:
        for shot in [1, 5, 10]:
            for seed in range(10):
                for method in ["zero_shot", "prototype", "logistic", "hybrid"]:
                    if method == "zero_shot" and "clip" not in model_name:
                        continue
                    if method == "hybrid" and "clip" not in model_name:
                        continue
                    all_combinations.append((model_name, centred, shot, seed, method))

for model_name, centred, shot, seed, method in tqdm(all_combinations, desc="Evaluating"):
    metrics = evaluate_method(
        model_name, method, shot, seed, centred,
        embeddings, text_embeddings
    )

    if metrics:
        results.append({
            "model": model_name,
            "method": method,
            "centred": centred,
            "shot": shot,
            "seed": seed,
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"]
        })

        # Compute per-class recall
        for cls in MAIN_CLASSES:
            cls_idx = class_to_idx[cls]
            cls_mask = metrics["true_labels"] == cls_idx
            if np.sum(cls_mask) > 0:
                cls_recall = np.sum((metrics["preds"] == cls_idx) & cls_mask) / np.sum(cls_mask)
                per_class_recalls.append({
                    "model": model_name,
                    "method": method,
                    "centred": centred,
                    "shot": shot,
                    "seed": seed,
                    "class": cls,
                    "recall": cls_recall,
                    "eval_count": np.sum(cls_mask)
                })

# Save summary of runs
df_results = pd.DataFrame(results)
df_recalls = pd.DataFrame(per_class_recalls)

df_results.to_csv(f"{RESULTS_ROOT}/summary_all_runs.csv", index=False)
df_recalls.to_csv(f"{RESULTS_ROOT}/recalls_all_runs.csv", index=False)

print("Done! Evaluation complete. Outputs written to:")
print(f"  - {RESULTS_ROOT}/summary_all_runs.csv")
print(f"  - {RESULTS_ROOT}/recalls_all_runs.csv")


Evaluating:   0%|          | 0/600 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

Done! Evaluation complete. Outputs written to:
  - /content/drive/MyDrive/eyeclip/results/summary_all_runs.csv
  - /content/drive/MyDrive/eyeclip/results/recalls_all_runs.csv


## Step 6: Summary Statistics & Confidence Intervals
Aggregating the experimental run logs and computing performance metrics along with Wilson score confidence intervals (Cell 6).

In [37]:
# Cell 6:computing performance metrics

import pandas as pd
R = "/content/drive/MyDrive/eyeclip/results/"
s = pd.read_csv(R + "summary_all_runs.csv"); r = pd.read_csv(R + "recalls_all_runs.csv")
print(s.shape, list(s.columns)); print(s.head(3).to_string()); print(r.shape, list(r.columns))
num = [c for c in s.columns if s[c].dtype.kind == "f"]
keys = [c for c in s.columns if c not in num and c != "seed"]
print(s.groupby(keys)[num].agg(["mean", "std"]).round(3).to_string())


import pandas as pd, numpy as np
R = "/content/drive/MyDrive/eyeclip/results/"
r = pd.read_csv(R + "recalls_all_runs.csv")
sel = r[(r.centred == True) & (r.shot == 10) & (r.method.isin(["logistic", "prototype", "hybrid"]))]
t = sel.groupby(["model", "method", "class"]).agg(recall=("recall", "mean"), sd=("recall", "std"),
                                                    n_eval=("eval_count", "first")).round(3)
z = 1.96; p, n = t.recall, t.n_eval   # approximate Wilson interval; ignores seed dependence
den = 1 + z*z/n; ctr = p + z*z/(2*n); w = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
t["ci_lo"] = ((ctr - w)/den).round(2); t["ci_hi"] = ((ctr + w)/den).round(2)
print(t.to_string())

(600, 7) ['model', 'method', 'centred', 'shot', 'seed', 'macro_f1', 'balanced_acc']
               model     method  centred  shot  seed  macro_f1  balanced_acc
0  resnet50_imagenet  prototype     True     1     0  0.738431      0.790801
1  resnet50_imagenet   logistic     True     1     0  0.749610      0.808658
2  resnet50_imagenet  prototype     True     1     1  0.609817      0.636472
(2400, 8) ['model', 'method', 'centred', 'shot', 'seed', 'class', 'recall', 'eval_count']
                                            macro_f1        balanced_acc       
                                                mean    std         mean    std
model                method    centred shot                                    
biomedclip_vit_b_16  hybrid    False   1       0.617  0.096        0.699  0.100
                                       5       0.689  0.054        0.783  0.035
                                       10      0.719  0.067        0.812  0.056
                               True   

## Step 7: Result Figure
Plots macro-F1 against 1/5/10 shots (mean +/- std over the 10 seeds) straight from `results/summary_all_runs.csv`,
so the figure can be rebuilt without re-running any model. Left: few-shot classifiers; right: zero-shot text prompts.

In [ ]:
# Cell 7: result figure from the saved CSV (no model re-run needed)
import pandas as pd, matplotlib.pyplot as plt

R = '/content/drive/MyDrive/eyeclip/results/'
s = pd.read_csv(R + 'summary_all_runs.csv')
s = s[s.centred]                      # the centred variant used in the README table

LABEL = {'clip_vit_b_16_openai': 'CLIP ViT-B/16', 'biomedclip_vit_b_16': 'BiomedCLIP',
         'resnet50_imagenet': 'ResNet-50 (baseline)'}
COLOR = {'clip_vit_b_16_openai': '#1f77b4', 'biomedclip_vit_b_16': '#2ca02c',
         'resnet50_imagenet': '#7f7f7f'}
STYLE = {'prototype': '--', 'logistic': '-', 'hybrid': ':'}
shots = [1, 5, 10]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={'width_ratios': [2.1, 1]})
ax = axes[0]
agg = (s[s.method != 'zero_shot'].groupby(['model', 'method', 'shot'])['macro_f1']
       .agg(['mean', 'std']).reset_index())
for (m, meth), g in agg.groupby(['model', 'method']):
    g = g.set_index('shot').reindex(shots)
    ax.plot(range(len(shots)), g['mean'], STYLE[meth], color=COLOR[m], marker='o', ms=4,
            lw=1.6, label=f'{LABEL[m]} - {meth}')
    ax.fill_between(range(len(shots)), g['mean'] - g['std'], g['mean'] + g['std'],
                    color=COLOR[m], alpha=0.12, lw=0)
ax.set_xticks(range(len(shots)), [f'{k}-shot' for k in shots])
ax.set_ylabel('macro-F1'); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
ax.set_title('Few-shot methods (mean +/- std, 10 seeds)')
ax.legend(fontsize=6.5, loc='lower right', framealpha=0.9)

ax2 = axes[1]
zs = s[s.method == 'zero_shot'].groupby('model')['macro_f1'].agg(['mean', 'std'])
models = [m for m in ['clip_vit_b_16_openai', 'biomedclip_vit_b_16'] if m in zs.index]
ax2.bar([LABEL[m] for m in models], zs.loc[models, 'mean'], yerr=zs.loc[models, 'std'],
        capsize=4, color=[COLOR[m] for m in models], alpha=0.85)
ax2.set_ylim(0, 1); ax2.set_ylabel('macro-F1'); ax2.grid(alpha=0.3, axis='y')
ax2.tick_params(axis='x', labelsize=8); ax2.set_title('Zero-shot (text prompts)', fontsize=10)

fig.suptitle('EyeCLIP pilot: macro-F1 vs labelled examples per class (4 classes, RFMiD 2.0)', fontsize=10)
fig.tight_layout()
fig.savefig(R + 'fig_fewshot.png', dpi=160)
print('wrote', R + 'fig_fewshot.png')
print(agg.pivot_table(index=['model', 'method'], columns='shot', values='mean').round(3).to_string())  # must match the README table

## Step 8: Per-image predictions, per-class metrics, bootstrap CIs, selective prediction, retrieval,
near-duplicate audit

Runs on the cached embeddings, same protocol as cells 5-7: logistic C=1 on the raw embeddings as the primary
classifier; prototype, hybrid and zero-shot on embeddings centred on the training-split mean; 5 and 10 shots;
seeds 0-9; all 3 models. Nothing is tuned on the evaluation set. Writes `predictions.csv`,
`per_class_prf.csv`, `bootstrap_ci.csv`, `fig_selective.png`, `retrieval.csv`, `retrieval_nearest.csv` and
`fig_similarity.png` to Drive (`/content/drive/MyDrive/eyeclip/results`), so the artifacts survive the Colab
session; set `WRITE_TO_REPO = True` in the cell to write into this checkout's `results/` instead. Bootstrap CIs
use the README estimator (macro-F1 per seed, then the mean over seeds).


In [ ]:
# Cell 8 — per-image predictions, per-class PRF, bootstrap CIs, selective prediction, retrieval,
# near-duplicate audit. Same protocol as cells 5-7: logistic C=1 on RAW embeddings as the primary
# classifier; prototype/hybrid on embeddings centred on the training-split mean. 10 seeds. Every
# setting below is fixed in advance; the eval set is never used to choose one.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

MAIN = ["WNL", "DR", "CRS", "RT"]; K = len(MAIN); CTI = {c: i for i, c in enumerate(MAIN)}
SHOTS, SEEDS = [5, 10], list(range(10))
C, SCALE, NBOOT, BOOT_SEED = 1.0, 100.0, 2000, 0   # logistic C; softmax(SCALE*cosine); bootstrap
PRIMARY, PRIMARY_SHOT = "logistic", 10
COVS = [1.0, 0.9, 0.8, 0.7, 0.5]
MODELS = ["clip_vit_b_16_openai", "biomedclip_vit_b_16", "resnet50_imagenet"]
LABEL = {"clip_vit_b_16_openai": "CLIP ViT-B/16", "biomedclip_vit_b_16": "BiomedCLIP",
         "resnet50_imagenet": "ResNet-50"}
COLOR = {"clip_vit_b_16_openai": "#1f77b4", "biomedclip_vit_b_16": "#2ca02c",
         "resnet50_imagenet": "#7f7f7f"}

# ── paths: cached embeddings and all outputs live on Drive, because a Colab runtime is thrown
# away at the end of the session and a `results/` in the checkout would go with it. Manifests
# come from this checkout when it has them. Set WRITE_TO_REPO=True to write into results/ here.
DRIVE = Path("/content/drive/MyDrive/eyeclip")
WRITE_TO_REPO = False
def man(f):
    p = Path("manifests") / f
    return p if p.exists() else DRIVE / "manifests" / f
RESULTS = Path("results") if WRITE_TO_REPO or not (DRIVE / "results").is_dir() else DRIVE / "results"
EMB = DRIVE / "embeddings" if (DRIVE / "embeddings").is_dir() else Path("embeddings")
RESULTS.mkdir(parents=True, exist_ok=True)
print("outputs ->", RESULTS)

manifest = pd.read_csv(man("manifest_cache.csv"))
sa = pd.read_csv(man("support_assignments.csv"))
emb = {m: np.load(EMB / f"{m}.npy") for m in MODELS}
txt = {m: np.load(EMB / f"{m}_text.npz") for m in MODELS if (EMB / f"{m}_text.npz").exists()}
METHODS = {m: (["logistic", "prototype", "hybrid"] if m in txt else ["logistic", "prototype"])
           for m in MODELS}

# ── evaluation set and labels (identical to cell 5) ──
ev = manifest.index[(manifest.single_class.isin(MAIN)) & (manifest.n_pos == 1) &
                    (manifest.orig_split.isin(["validation", "test"]))]
y = np.array([CTI[c] for c in manifest.loc[ev, "single_class"]])
stems = manifest.loc[ev, "stem"].astype(str).values
n = len(y)
train_idx = manifest.index[manifest.orig_split == "train"]
mu = {m: emb[m][train_idx].mean(0) for m in MODELS}     # centring mean: train split only

def prep(m, idx, centred=True):
    X = emb[m][idx] - mu[m] if centred else emb[m][idx]
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def support(shot, seed):
    """{class: [manifest row indices]} — same rows cell 5 draws, nested across shots."""
    s = sa[sa.seed == seed]
    return {r["class"]: [manifest.index[manifest.stem.astype(str) == q][0]
                         for q in str(r.support).split(";")[:shot]]
            for _, r in s.iterrows() if r["class"] in MAIN}

def softmax(S):
    S = SCALE * S; S = S - S.max(1, keepdims=True); P = np.exp(S)
    return P / P.sum(1, keepdims=True)

def proto_scores(m, sidx, Xe):
    P = np.stack([prep(m, sidx[c]).mean(0) for c in MAIN])
    return Xe @ (P / np.linalg.norm(P, axis=1, keepdims=True)).T

def oof_support_conf(m, flat, ysup, seed, folds=5):
    """Confidence on each support image from a model fitted without it (5-fold within the support set)."""
    conf = np.zeros(len(flat))
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for tr, te in skf.split(np.zeros(len(flat)), ysup):
        c = LogisticRegression(C=C, max_iter=1000, solver="lbfgs")
        conf[te] = c.fit(emb[m][flat][tr], ysup[tr]).predict_proba(emb[m][flat][te]).max(1)
    return conf

def text_scores(m, Xe):
    cls = list(txt[m]["classes"])
    T = txt[m]["emb"][[cls.index(c) for c in MAIN]]
    return Xe @ T.T

# ── 1) one row per (model, method, shot, seed, eval image) ────────────────────
Xe = {m: prep(m, ev) for m in MODELS}
PROB, SUPP_CONF, rows = {}, {}, []
for m in MODELS:
    for shot in SHOTS:
        for seed in SEEDS:
            sidx = support(shot, seed)
            flat = [i for c in MAIN for i in sidx[c]]
            ysup = np.array([CTI[c] for c in MAIN for _ in sidx[c]])
            clf = LogisticRegression(C=C, max_iter=1000, solver="lbfgs")
            # logistic is fitted on the raw cached embeddings, exactly as in cell 5,
            # so its numbers reproduce results/summary_all_runs.csv
            clf.fit(emb[m][flat], ysup)
            S = {}
            P = np.zeros((n, K)); P[:, clf.classes_] = clf.predict_proba(emb[m][ev])
            S["logistic"] = P
            S["prototype"] = softmax(proto_scores(m, sidx, Xe[m]))
            if m in txt:
                S["hybrid"] = softmax(0.5 * proto_scores(m, sidx, Xe[m]) + 0.5 * text_scores(m, Xe[m]))
            if shot == PRIMARY_SHOT:   # out-of-fold support confidences for the flagging rule
                SUPP_CONF[m, seed] = oof_support_conf(m, flat, ysup, seed)
            for meth in METHODS[m]:
                P = S[meth]; PROB[m, meth, shot, seed] = P
                pred = P.argmax(1)
                d = pd.DataFrame({"model": m, "method": meth, "shot": shot, "seed": seed,
                                  "stem": stems, "true_class": [MAIN[i] for i in y],
                                  "pred_class": [MAIN[i] for i in pred]})
                for j, c in enumerate(MAIN): d[f"score_{c}"] = P[:, j]
                rows.append(d)

pred_df = pd.concat(rows, ignore_index=True)
pred_df.to_csv(RESULTS / "predictions.csv", index=False)
print(f"predictions.csv: {pred_df.shape[0]} rows, {pred_df.shape[1]} cols")

# ── 2) confusion at 10 shots (mean counts over seeds) + per-class PRF ─────────
print("\n── confusion at 10 shots, counts averaged over 10 seeds (rows=true, cols=pred) ──")
for m in MODELS:
    for meth in METHODS[m]:
        Cm = np.mean([confusion_matrix(y, PROB[m, meth, 10, s].argmax(1), labels=range(K))
                      for s in SEEDS], 0)
        print(f"\n{m} | {meth}")
        print(pd.DataFrame(Cm, index=MAIN, columns=MAIN).round(1).to_string())

AVG = {(k[0], k[1]): np.mean([PROB[k[0], k[1], 10, s] for s in SEEDS], 0)
       for k in PROB if k[2] == 10}
prf_rows, prf_tables = [], {}
for (m, meth), P in AVG.items():
    yh = P.argmax(1)
    p, r, f, sup = precision_recall_fscore_support(y, yh, labels=range(K), zero_division=0)
    for j, c in enumerate(MAIN):
        prf_rows.append({"model": m, "method": meth, "class": c, "n_eval": int(sup[j]),
                         "precision": p[j], "recall": r[j], "f1": f[j]})
    prf_tables[m, meth] = pd.DataFrame(
        {"precision": p, "recall": r, "f1": f, "n_eval": sup.astype(int)}, index=MAIN).round(3)
pd.DataFrame(prf_rows).to_csv(RESULTS / "per_class_prf.csv", index=False)
print("\n── per-class precision / recall / F1, 10 shots (one prediction per image = "
      "argmax of the seed-averaged scores) ──")
for (m, meth), t in prf_tables.items():
    print(f"\n{m} | {meth}")
    print(t.to_string())

# ── 3) bootstrap over eval images: macro-F1 CI and paired model differences ───
# Estimator matches the README table: macro-F1 per seed, then the mean over the 10 seeds.
# Per resample the same 10 seed-level scores are averaged, so the point estimates equal the
# README means and each CI is on the same scale as the table.
def macro_f1(yt, yp):
    """Macro-F1 over the fixed 4 labels (classes absent from a resample score 0)."""
    Mx = np.bincount(yt * K + yp, minlength=K * K).reshape(K, K)
    tp = np.diag(Mx); fp = Mx.sum(0) - tp; fn = Mx.sum(1) - tp
    den = 2 * tp + fp + fn
    return np.divide(2 * tp, den, out=np.zeros(K), where=den > 0).mean()

YHAT = {m: [PROB[m, PRIMARY, 10, s].argmax(1) for s in SEEDS] for m in MODELS}
rng = np.random.default_rng(BOOT_SEED)
idx = rng.integers(0, n, size=(NBOOT, n))          # one resample shared by every model
boot = {m: np.array([np.mean([macro_f1(y[i], yh[i]) for yh in YHAT[m]]) for i in idx])
        for m in MODELS}
point = {m: np.mean([macro_f1(y, yh) for yh in YHAT[m]]) for m in MODELS}
b_rows = [{"comparison": m, "point": point[m],
           "ci_lo": np.percentile(boot[m], 2.5), "ci_hi": np.percentile(boot[m], 97.5)}
          for m in MODELS]
for name, a, b in [("clip_minus_resnet", "clip_vit_b_16_openai", "resnet50_imagenet"),
                   ("biomedclip_minus_resnet", "biomedclip_vit_b_16", "resnet50_imagenet")]:
    d = boot[a] - boot[b]
    b_rows.append({"comparison": name, "point": point[a] - point[b],
                   "ci_lo": np.percentile(d, 2.5), "ci_hi": np.percentile(d, 97.5)})
boot_df = pd.DataFrame(b_rows)
boot_df.to_csv(RESULTS / "bootstrap_ci.csv", index=False)
print(f"\n── macro-F1, logistic 10-shot, {NBOOT} bootstrap resamples of the {n} eval images "
      f"(95% percentile CI) ──")
print(boot_df.round(3).to_string(index=False))

# ── 4) uncertainty and selective prediction (primary config, 10 shots) ────────
print("\n── logistic 10-shot: mean confidence / top-2 margin / entropy over seeds ──")
u_rows = []
for m in MODELS:
    cf, mg, en = [], [], []
    for s in SEEDS:
        P = PROB[m, PRIMARY, 10, s]; q = np.sort(P, 1)
        cf.append(P.max(1).mean()); mg.append((q[:, -1] - q[:, -2]).mean())
        en.append((-(P * np.log(np.clip(P, 1e-12, 1))).sum(1)).mean())
    u_rows.append({"model": m, "confidence": np.mean(cf), "margin": np.mean(mg),
                   "entropy": np.mean(en)})
print(pd.DataFrame(u_rows).round(3).to_string(index=False))

sel = {k: [] for k in ("macro_f1", "accuracy")}
for m in MODELS:
    for cov in COVS:
        f1s, accs = [], []
        for s in SEEDS:
            P = PROB[m, PRIMARY, 10, s]
            keep = np.argsort(-P.max(1))[:int(round(cov * n))]
            f1s.append(macro_f1(y[keep], P.argmax(1)[keep]))
            accs.append((y[keep] == P.argmax(1)[keep]).mean())
        sel["macro_f1", m, cov] = np.mean(f1s); sel["accuracy", m, cov] = np.mean(accs)
print("\n── selective prediction, logistic 10-shot (mean over seeds; coverage = kept fraction) ──")
for metric in ("macro_f1", "accuracy"):
    t = pd.DataFrame({LABEL[m]: [sel[metric, m, c] for c in COVS] for m in MODELS},
                     index=[f"{int(c*100)}%" for c in COVS])
    print(f"\n{metric}" + "\n" + t.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, metric in zip(axes, ("macro_f1", "accuracy")):
    for m in MODELS:
        ax.plot([c * 100 for c in COVS], [sel[metric, m, c] for c in COVS], marker="o", ms=4,
                lw=1.6, color=COLOR[m], label=LABEL[m])
    ax.set_xlabel("coverage (%)"); ax.set_ylabel(metric); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7); fig.suptitle("Selective prediction (logistic, 10-shot, mean of 10 seeds)", fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS / "fig_selective.png", dpi=160); plt.close(fig)
print(f"\nwrote {RESULTS / 'fig_selective.png'}")

# Flagging rule: eval confidence below the 20th percentile of the SUPPORT-set confidences.
# The support confidences are out-of-fold (5-fold CV inside the 40-image support set), so unlike an
# in-sample threshold this one is not optimistic. The curve above is the main result; this count is
# a secondary, illustrative summary.
print("\n── illustrative: images flagged by 'confidence < 20th percentile of support confidences' "
      f"(eval n={n}; support confidences are 5-fold out-of-fold) ──")
flag = []
for m in MODELS:
    cnt = [int((PROB[m, PRIMARY, 10, s].max(1) < np.percentile(SUPP_CONF[m, s], 20)).sum())
           for s in SEEDS]
    flag.append({"model": m, "flagged_mean": np.mean(cnt), "min": min(cnt), "max": max(cnt),
                 "pct_mean": 100 * np.mean(cnt) / n})
print(pd.DataFrame(flag).round(1).to_string(index=False))
print("note: illustrative only — the selective-prediction curve is the main evidence, and this count "
      "is not a calibrated probability (similarity scores are not calibrated)")

# ── 5) retrieval: top-5 cosine neighbours of each eval image among train images ──
pool = manifest.index[(manifest.orig_split == "train") & (manifest.n_pos == 1) &
                      (manifest.single_class.isin(MAIN))]
ypool = np.array([CTI[c] for c in manifest.loc[pool, "single_class"]])
ret, nn_rows = [], []
for m in MODELS:
    S = prep(m, ev) @ prep(m, pool).T
    hit = ypool[np.argsort(-S, 1)[:, :5]] == y[:, None]     # (n, 5) correct-class indicators
    top1 = S.argmax(1)
    nn_rows.append(pd.DataFrame({"model": m, "stem": stems, "true_class": [MAIN[i] for i in y],
                                 "neighbour_stem": manifest.loc[pool, "stem"].astype(str).values[top1],
                                 "neighbour_class": [MAIN[i] for i in ypool[top1]],
                                 "cosine": S[np.arange(n), top1]}))
    ret.append({"model": m, "class": "ALL", "n": n, "precision_at_1": hit[:, 0].mean(),
                "precision_at_5": hit.mean(1).mean()})
    for j, c in enumerate(MAIN):
        k = y == j
        ret.append({"model": m, "class": c, "n": int(k.sum()), "precision_at_1": hit[k, 0].mean(),
                    "precision_at_5": hit[k].mean(1).mean()})
ret_df = pd.DataFrame(ret)
ret_df.to_csv(RESULTS / "retrieval.csv", index=False)
print(f"\n── retrieval, top-5 cosine neighbours in the {len(pool)}-image train pool "
      "(chance P@1 ≈ class share) ──")
print(ret_df.round(3).to_string(index=False))

# ── near-duplicate audit, free with the retrieval pass: how close is each eval image to its
# nearest training image? High similarity can flag a possible duplicate or same-patient pair;
# it cannot prove either (no patient IDs are used).
nn = pd.concat(nn_rows, ignore_index=True)
nn.to_csv(RESULTS / "retrieval_nearest.csv", index=False)
print(f"\n── near-duplicate audit: top-1 cosine of each eval image to the {len(pool)}-image train pool"
      f" (wrote {RESULTS / 'retrieval_nearest.csv'}) ──")
print(nn.groupby("model").cosine.agg(n="size", median="median", p99=lambda s: s.quantile(0.99),
                                      max="max", above_0_99=lambda s: int((s > 0.99).sum())
                                      ).round(3).to_string())

fig, axes = plt.subplots(1, len(MODELS), figsize=(9, 2.8), sharex=True, sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODELS):
    s = nn[nn.model == m]
    for c in MAIN:
        ax.hist(s.cosine[s.true_class == c], bins=20, range=(s.cosine.min(), 1.0), alpha=0.45,
                density=True, label=c)
    ax.set_title(LABEL[m], fontsize=9); ax.set_xlabel("top-1 cosine to train pool"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("density")
np.atleast_1d(axes)[-1].legend(fontsize=7)
fig.suptitle("Eval-to-train nearest-neighbour similarity by true class", fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS / "fig_similarity.png", dpi=160); plt.close(fig)

print("\n── 10 highest-similarity eval/train pairs (CLIP ViT-B/16; sorted by cosine) ──")
top = nn[nn.model == "clip_vit_b_16_openai"].nlargest(10, "cosine")
print(top[["stem", "true_class", "neighbour_stem", "neighbour_class", "cosine"]]
      .assign(cosine=lambda d: d.cosine.round(4)).to_string(index=False))
print(f"wrote {RESULTS / 'fig_similarity.png'}")
